# Scientific Programming with Python 
## (Winter 2025/26)

# Session 05: MatPlotLib II

* Some more python: generators
* MatPlotLib

# Python: Generators

* Introduced in [PEP 255](https://peps.python.org/pep-0255/)
* a `generator` can generate values
* special type of iterator
* Generators can be realized in two ways:
  - generator function
  - generator expressions

## Generator functions

Python generator function is a function which returns a generator.
* Generator functions are implicitly defined by the use of `yield` in the function body.
* `yield` may be used with a value, in which case that value is treated as the "generated" value.
* The next time `next()` is called on the generator (i.e. in the next step in a for loop, for example), the generator resumes execution from where it called `yield`, not from the beginning of the function.
* All of the state, like the values of local variables, is recovered and the generator contiues to execute until the next call to `yield`. 

In [ ]:
def generate_numbers():
    yield 1
    yield 10
    yield 3
    yield 5

In [ ]:
my_generator = generate_numbers()

In [ ]:
type(my_generator)

In [ ]:
next(my_generator)

In [ ]:
next(my_generator)

In [ ]:
next(my_generator)

In [ ]:
next(my_generator)

In [ ]:
next(my_generator)

In [ ]:
for number in generate_numbers():
    print(number)

### The `yield` statement

* use `yield` similar to `return` in a function
* when the Python yield statement is hit, the program suspends function execution and returns the yielded value to the caller
  - in contrast, `return` stops function execution completely
* when suspended, the state of that function is saved
  - this includes variable bindings local to the generator, the instruction pointer, the internal stack, and any exception handling
* allows you to resume function execution whenever you call one of the generator’s methods
  - evaluation picks back up right after `yield`
* there can be multiple `yield`statements in a function
  - typically `yield` is nested in some loop

### Example: an infinite sequence generator

In [ ]:
def infinite_sequence():
    num = 0
    while True:
        yield num
        num += 1

In [ ]:
gen = infinite_sequence()

Its type is `generator`:

In [ ]:
type(gen)

The generator can be used like a normal iterator:

In [ ]:
next(gen)

In [ ]:
next(gen)

In [ ]:
next(gen)

In [ ]:
for i in infinite_sequence():
    print(i, end=" ")
    if i >= 100:
        break

## Generator expressions

* generator expressions allow you to quickly create a generator object
* similar to list comprehensions
* advantage: generator expressions do not require to hold all data in memory

In [ ]:
# list comprehension
squares_list = [num**2 for num in range(5)]

type(squares_list)

In [ ]:
# generator expression
squares_generator = (num**2 for num in range(5))

type(squares_generator)

In [ ]:
squares_list

In [ ]:
squares_generator

In [ ]:
len(squares_list)

In [ ]:
len(squares_generator)

In [ ]:
squares_list[2]

In [ ]:
squares_generator[2]

In [ ]:
for number in squares_list:
    print(number)

In [ ]:
for number in squares_generator:
    print(number)

List store all of their elements in memeory:

In [ ]:
import sys

squares_list = [i ** 2 for i in range(10000)]
sys.getsizeof(squares_list)

Generators only generate objects on demand and hence do not need to store them:

In [ ]:
squares_generator = (i ** 2 for i in range(10000))
print(sys.getsizeof(squares_generator))

## Advanced Generator Methods

In addition to `yield`, generator objects can make use of the following methods:
* `.send()`
* `.throw()`
* `.close()`

The `.send()`-Method allows to send values to the generator:

In [ ]:
def echo():
    while True:
        value = yield  # Wait for a value to be sent
        print(f"Received: {value}")

In [ ]:
# Create a generator
generator = echo()

# Start the generator
next(generator)

In [ ]:
# Send values to the generator
generator.send("Hello")

In [ ]:
generator.send("World")

In [ ]:
next(generator)

Use Cases for the send Function:

* Coroutines: Coroutines are a generalization of generators that allow for cooperative multitasking. Using `send`, allows to create coroutines that perform complex tasks by receiving input at specific points in their execution.
* Pipelines: in data processing pipelines, the `send` function can be used to pass data between different stages of the pipeline, allowing for modular and flexible designs.
* State Management: manage the state in your application by allowing the generator to respond to external events or commands, thus controlling its behavior dynamically.

In [ ]:
def accumulator():
    total = 0
    while True:
      	# Yield the current total
        value = yield total
        # Add the received value to the total
        total += value

# Create the coroutine
acc = accumulator()

# Start the coroutine
next(acc)

In [ ]:
print(acc.send(5))

In [ ]:
print(acc.send(10))

In [ ]:
print(acc.send(20))

The `throw` method allows to pass an Exception to a generator.  The excepton could then be handled inside the generator:

In [ ]:
generator.throw(KeyboardInterrupt)

The `.close` method allows to prematurely end a generator.  Especially useful with infinite iterators.

In [ ]:
# Create a generator
generator = echo()

# Start the generator
next(generator)

In [ ]:
generator.send("Hello")

In [ ]:
generator.close()

In [ ]:
generator.send("Hello")

# Anatomy of a "Plot"
Matplotlib is a large project and can seem daunting at first. However, by learning the components, it should begin to feel much smaller and more approachable.

People use "plot" to mean many different things.  Here, we'll be using a consistent terminology (mirrored by the names of the underlying classes, etc):

<img src="images/figure_axes_axis_labeled.png">

* The ``Figure`` is the top-level container in this hierarchy.
  - It is the overall window/page that everything is drawn on.
  - You can have multiple independent figures and ``Figure``s can contain multiple ``Axes``. 

* Most plotting ocurs on an ``Axes``.
  - The axes is effectively the area that we plot data on and any ticks/labels/etc associated with it.
  - Usually we'll set up an Axes with a call to ``subplot``
  - `subplot` places Axes on a regular grid, so in most cases, ``Axes`` and ``Subplot`` are synonymous.

* Each ``Axes`` has an ``XAxis`` and a ``YAxis``.
  - These contain the ticks, tick locations, labels, etc.
  - we will usually not touch ``Axis`` directly, instead use other function to control ticks, tick labels, and data limits
  - however, it is worth mentioning here to explain where the term ``Axes`` comes from.

## Working with `Figures` and `Axes`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
fig = plt.figure()

Awww, nothing happened! This is because by default mpl will not show anything until told to do so, as we mentioned earlier in the "backend" discussion.

Instead, we'll need to call ``plt.show()``

In [ ]:
plt.show()

Still nothing. That is because the notebook does not want to show empty figures. In a plain python script, you would actually get an empty figure here. To actually see the figure in the notebook, we need to add some axes...

## Axes

All plotting is done with respect to an [`Axes`](http://matplotlib.org/api/axes_api.html#matplotlib.axes.Axes). An *Axes* is made up of [`Axis`](http://matplotlib.org/api/axis_api.html#matplotlib.axis.Axis) objects and many other things. An *Axes* object must belong to a *Figure* (and only one *Figure*). Most commands you will ever issue will be with respect to this *Axes* object.

Typically, you'll set up a `Figure`, and then add an `Axes` to it. 

You can use `fig.add_axes`, but in most cases, you'll find that adding a subplot will fit your needs perfectly. (Again a "subplot" is just an axes on a grid system.) 

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(221)
ax2 = fig.add_subplot(224)
plt.show()

You can control the size of the figure through the ``figsize`` argument, which expects a tuple of ``(width, height)`` in inches. 

BTW: A really useful utility function is [`figaspect`](http://matplotlib.org/api/figure_api.html?highlight=figaspect#matplotlib.figure.figaspect)

In [ ]:
fig = plt.figure(figsize=(10, 5))  # Twice as high as wide.
ax = fig.add_subplot(111)
plt.show()

Matplotlib's objects typically have lots of "explicit setters" -- in other words, functions that start with ``set_<something>`` and control a particular option. 

To demonstrate this (and as an example of IPython's tab-completion), try typing `ax.set_` in a code cell, then hit the `<Tab>` key.  You'll see a long list of `Axes` methods that start with `set`.

For example, we could have written the third line above as:

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
ax.set_xlim([0.5, 4.5])
ax.set_ylim([-2, 8])
ax.set_title("An Example Axes")
ax.set_ylabel("Y-Axis")
ax.set_xlabel("X-Axis")
plt.show()

Clearly this can get repitive quickly.  Therefore, Matplotlib's `set` method can be very handy.  It takes each kwarg you pass it and tries to call the corresponding "setter".  For example, `ax.set(foo='bar')` would call `ax.set_foo('bar')`.

Note that the `set` method doesn't just apply to `Axes`; it applies to more-or-less all matplotlib objects.

However, there are cases where you'll want to use things like `ax.set_xlabel('Some Label', size=25)` to control other options for a particular function.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
ax.set(
    xlim=[0.5, 4.5], 
    ylim=[-2, 8], 
    title='An Example Axes',
    ylabel='Y-Axis', 
    xlabel='X-Axis',
)
plt.show()

## Axes methods (object oriented interface) vs. pyplot (state machine interface)

Up to now we used the `matplotlib.pyplot` functions for plotting.  These function implicitly create an `Axes` object and implicitly calls methods of that object.
Just about all methods of an `Axes` object exist as a function in the `pyplot` module (and vice-versa). 

For example, when calling `plt.xlim(1, 10)`, `pyplot` calls `ax.set_xlim(1, 10)` on whichever `Axes` is *current*. Here is an equivalent version of the above example using just `pyplot`.

In [ ]:
plt.figure()
plt.plot([10, 20, 25, 35])
plt.show()

The same plot can be obtained using the object oriented `Axes` interface:

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
ax.plot([10, 20, 25, 35])
plt.show()

In general (except for very simple plots) it is preferable to uses the `Axes` interface:
* when doing more complicated plots, or working within larger scripts, you will want to explicitly pass around the *Axes* and/or *Figure* object to operate upon.
* The advantage of keeping which axes we're working with very clear in our code will become more obvious when we start to have multiple axes in one figure.
* [PEP20](http://www.python.org/dev/peps/pep-0020/) "The Zen of Python" says: "Explicit is better than implicit"

## Multiple Axes

A figure can have more than one `Axes` on it.  If you want your axes to be on a regular grid system, then it's easiest to use `plt.subplots(...)` to create a figure and add the axes to it automatically.

For example:

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2)
plt.show()

In [ ]:
axes

`plt.subplots(...)` created a new figure and added 4 subplots to it.  The `axes` object that was returned is a 2D numpy object array.  Each item in the array is one of the subplots.  They're laid out as you see them on the figure.  

Therefore, when we want to work with one of these axes, we can index the `axes` array and use that item's methods.

For example:

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2)
axes[0, 0].set(title="Upper Left")
axes[0, 1].set(title="Upper Right")
axes[1, 0].set(title="Lower Left")
axes[1, 1].set(title="Lower Right")

# tight_layout makes sure titles and tick labels do not overlap.
fig.tight_layout()

plt.show()

One really nice thing about `plt.subplots()` is that when it's called with no arguments, it creates a new figure with a single subplot. 

Any time you see something like

```
fig = plt.figure()
ax = fig.add_subplot(111)
```

You can replace it with:

```
fig, ax = plt.subplots()
```

We'll be using that approach for the rest of the examples.  It's much cleaner.  

However, keep in mind that we're still creating a figure and adding axes to it.  When we start making plot layouts that can't be described by `subplots`, we'll go back to creating the figure first and then adding axes to it one-by-one.

In [ ]:
fig, ax = plt.subplots()

type(ax)

Speaking of titles, you can also set a supertitle for an entire figure.

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2)
axes[0, 0].set(title="Upper Left")
axes[0, 1].set(title="Upper Right")
axes[1, 0].set(title="Lower Left")
axes[1, 1].set(title="Lower Right")

# tight_layout makes sure titles and tick labels do not overlap.
fig.tight_layout()
fig.suptitle("Four subplots")

plt.show()

## Subplot Spacing
The spacing between the subplots can be adjusted using [`fig.subplots_adjust()`](http://matplotlib.org/api/pyplot_api.html?#matplotlib.pyplot.subplots_adjust). Play around with the example below to see how the different arguments affect the spacing.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 9))
fig.subplots_adjust(wspace=1.3, hspace=0.7,
                   left=0.425, right=0.8,
                   top=0.7,    bottom=0.2)
plt.show()

A common "gotcha" is that the labels are not automatically adjusted to avoid overlapping those of another subplot. Matplotlib does not currently have any sort of robust layout engine, as it is a design decision to minimize the amount of "magical plotting". We intend to let users have complete, 100% control over their plots. LaTeX users would be quite familiar with the amount of frustration that can occur with automatic placement of figures in their documents.

That said, there have been some efforts to develop tools that users can use to help address the most common compaints. The "[Tight Layout](http://matplotlib.org/users/tight_layout_guide.html)" feature, when invoked, will attempt to resize margins and subplots so that nothing overlaps.

If you have multiple subplots, and want to avoid overlapping titles/axis labels/etc, `fig.tight_layout` is a great way to do so:

In [ ]:
def example_plot(ax):
    ax.plot([1, 2])
    ax.set_xlabel('x-label', fontsize=16)
    ax.set_ylabel('y-label', fontsize=8)
    ax.set_title('Title', fontsize=24)

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(nrows=2, ncols=2)
example_plot(ax1)
example_plot(ax2)
example_plot(ax3)
example_plot(ax4)

plt.show()

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(nrows=2, ncols=2)
example_plot(ax1)
example_plot(ax2)
example_plot(ax3)
example_plot(ax4)

# Tight layout enabled.
fig.tight_layout()

plt.show()

# How to speak "MPL"

Control plots and figures: the substance and vocabulary of the library. 

## Colors
This is, perhaps, the most important piece of vocabulary in Matplotlib. Given that Matplotlib is a plotting library, colors are associated with everything that is plotted in your figures. Matplotlib supports a [very robust language](http://matplotlib.org/api/colors_api.html#module-matplotlib.colors) for specifying colors that should be familiar to a wide variety of users.

By default matplotlib will choose different colors when combining data on the same axes.

In [ ]:
t = np.arange(0.0, 5.0, 0.2)
fig, ax = plt.subplots()
ax.plot(t, t, linewidth=5)
ax.plot(t, t**2, linewidth=5)
ax.plot(t, t**3, linewidth=5)
plt.show()

### Colornames
First, colors can be given as strings. For very basic colors, you can even get away with just a single letter:

- b: blue
- g: green
- r: red
- c: cyan
- m: magenta
- y: yellow
- k: black
- w: white

Other colornames that are allowed are the HTML/CSS colornames such as "burlywood" and "chartreuse". See the [full list](https://www.w3schools.com/colors/colors_names.asp) of the 147 colornames.

### Hex values
Colors can also be specified by supplying a HTML/CSS hex string, such as `'#0000FF'` for blue. Support for an optional alpha channel was added for v2.0. For more information about hex colors have a look at https://en.wikipedia.org/wiki/Web_colors#Hex_triplet.

In [ ]:
t = np.arange(0.0, 5.0, 0.2)
fig, ax = plt.subplots()
ax.plot(t, t, linewidth=5, color='AliceBlue')
ax.plot(t, t**2, linewidth=5, color='#ff00ff')
ax.plot(t, t**3, linewidth=5, color='#ffcc00')
plt.show()

### 256 Shades of Gray
A gray level can be given instead of a color by passing a string representation of a number between 0 and 1 (inclusive). `'0.0'` is black, while `'1.0'` is white. `'0.75'` would be a light shade of gray.


In [ ]:
t = np.arange(0.0, 5.0, 0.2)
fig, ax = plt.subplots()
ax.plot(t, t, linewidth=5, color='.9')
ax.plot(t, t**2, linewidth=5, color='0.5')
ax.plot(t, t**3, linewidth=5, color='0.0')
plt.show()

### RGB[A] tuples
You may come upon instances where the previous ways of specifying colors do not work. This can sometimes happen in some of the deeper, stranger levels of the library. When all else fails, the universal language of colors for matplotlib is the RGB[A] tuple. This is the "Red", "Green", "Blue", and sometimes "Alpha" tuple of floats in the range of [0, 1]. One means full saturation of that channel, so a red RGBA tuple would be `(1.0, 0.0, 0.0, 1.0)`, whereas a partly transparent green RGBA tuple would be `(0.0, 1.0, 0.0, 0.75)`.  The documentation will usually specify whether it accepts RGB or RGBA tuples. Sometimes, a list of tuples would be required for multiple colors, and you can even supply a Nx3 or Nx4 numpy array in such cases.

In functions such as `plot()` and `scatter()`, while it may appear that they can take a color specification, what they really need is a "format specification", which includes color as part of the format. Unfortunately, such specifications are string only and so RGB[A] tuples are not supported for such arguments (but you can still pass an RGB[A] tuple for a "color" argument).

Oftentimes there is a separate argument for "alpha" where-ever you can specify a color. The value for "alpha" will usually take precedence over the alpha value in the RGBA tuple. There is no easy way around this inconsistency.

In [ ]:
t = np.arange(0.0, 3.0, 0.2)
fig, ax = plt.subplots()
ax.plot(t, t, linewidth=5, color=(0, 0, 1))
ax.plot(t, t**2, linewidth=5, color=(0, 0.5, 0.5, 0.1))
ax.plot(t, t**3, linewidth=5, color=(0, 0, 1, 0.3))
# the alpha value can also be specified as an additional kwarg
ax.plot(t, t**4, linewidth=5, color=(0, 1, 1), alpha=.1)
plt.show()

### Cycle references
With the advent of fancier color cycles coming from the many available styles, users needed a way to reference those colors in the style without explicitly knowing what they are. So, in v2.0, the ability to reference the first 10 iterations of the color cycle was added. Whereever one could specify a color, you can supply a 2 character string of 'C#'. So, 'C0' would be the first color, 'C1' would be the second, and so on and so forth up to 'C9'.

In [ ]:
t = np.arange(0.0, 5.0, 0.2)
fig, ax = plt.subplots()
ax.plot(t, t, linewidth=5)
ax.plot(t, t**2, linewidth=5)
ax.plot(t, t**3, linewidth=5)
plt.show()

In [ ]:
t = np.arange(0.0, 5.0, 0.2)
fig, ax = plt.subplots()
ax.plot(t, t, linewidth=5, color='C2')
ax.plot(t, t**2, linewidth=5, color='C0')
ax.plot(t, t**3, linewidth=5, color='C0')
plt.show()

## Markers
[Markers](http://matplotlib.org/api/markers_api.html) are commonly used in [`plot()`](http://matplotlib.org/api/pyplot_api.html#matplotlib.pyplot.plot) and [`scatter()`](http://matplotlib.org/api/pyplot_api.html#matplotlib.pyplot.scatter) plots, but also show up elsewhere. There is a wide set of markers available, and custom markers can even be specified.

marker     |  description  | marker    |  description    |marker    |  description  | marker    |  description  
:----------|:--------------| :---------|:--------------  |:---------|:--------------| :---------|:--------------
"."        |  point        | "+"       |  plus           |","       |  pixel        | "x"       |  cross
"o"        |  circle       | "D"       |  diamond        |"d"       |  thin_diamond |           |
"8"        |  octagon      | "s"       |  square         |"p"       |  pentagon     | "\*"      |  star
"&#124;"   |  vertical line| "\_"      | horizontal line | "h"      |  hexagon1     | "H"       |  hexagon2
0          |  tickleft     | 4         |  caretleft      |"<"       | triangle_left | "3"       |  tri_left
1          |  tickright    | 5         |  caretright     |">"       | triangle_right| "4"       |  tri_right
2          |  tickup       | 6         |  caretup        |"^"       | triangle_up   | "2"       |  tri_up
3          |  tickdown     | 7         |  caretdown      |"v"       | triangle_down | "1"       |  tri_down
"None"     |  nothing      | `None`    |  default        |" "       |  nothing      |""         |  nothing


In [ ]:
t = np.arange(0.0, 5.0, 0.2)
fig, ax = plt.subplots()
ax.plot(t, t, '.', linewidth=5)
ax.plot(t, t**2, 'o', linewidth=5)
ax.plot(t, t**3, marker='+', linewidth=1) # With explicit arguments, you can set maker and linestyle separately.
ax.plot(t, -t, ls='', marker='v', linewidth=5) 
plt.show()

## Linestyles
Line styles are about as commonly used as colors. There are a few predefined linestyles available to use. Note that there are some advanced techniques to specify some custom line styles. [Here](http://matplotlib.org/1.3.0/examples/lines_bars_and_markers/line_demo_dash_control.html) is an example of a custom dash pattern.

linestyle          | description
-------------------|------------------------------
'-'                | solid
'--'               | dashed
'-.'               | dashdot
':'                | dotted
'None'             | draw nothing
' '                | draw nothing
''                 | draw nothing

Also, don't mix up ".-" (line with dot markers) and "-." (dash-dot line) when using the ``plot`` function!

In [ ]:
t = np.arange(0.0, 5.0, 0.2)
fig, ax = plt.subplots()
ax.plot(t, t, linestyle='-', linewidth=5)
ax.plot(t, t**2, linestyle='--', linewidth=5)
ax.plot(t, t**3, linestyle='-.', linewidth=5)
ax.plot(t, -t, linestyle=':', linewidth=5)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1)
ax.bar([1, 2, 3, 4], [10, 20, 15, 13], linestyle='--', edgecolor='r', linewidth=4)
plt.show()

## Colormaps
Another very important property of many figures is the colormap. The job of a colormap is to relate a scalar value to a color. In addition to the regular portion of the colormap, an "over", "under" and "bad" color can be optionally defined as well. NaNs will trigger the "bad" part of the colormap.

As we all know, we create figures in order to convey information visually to our readers. There is much care and consideration that have gone into the design of these colormaps. Your choice in which colormap to use depends on what you are displaying. In mpl, the "jet" colormap has historically been used by default, but it will often not be the colormap you would want to use. Much discussion has taken place on the mailing lists with regards to what colormap should be default. The v2.0 release of Matplotlib adopted a new default colormap, 'viridis', along with some other stylistic changes to the defaults.

[Here is the talk](https://www.youtube.com/watch?v=xAoljeRJ3lU) by Nathaniel Smith and Stéfan van der Walt at SciPy 2015 that does an excelent job explaining colormaps and how the new perceptual uniform colormaps where designed.



In [ ]:
def plot_cmap(name, value_range=(0, 1)):
    gradient = np.linspace(*value_range, 256)
    gradient = np.vstack((gradient, gradient))
    fig, ax = plt.subplots(figsize=plt.figaspect(0.1))
    ax.imshow(gradient, aspect='auto', cmap=plt.get_cmap(name), vmin=0, vmax=1)
    pos = list(ax.get_position().bounds)
    x_text = pos[0] - 0.01
    y_text = pos[1] + pos[3]/2.
    ax.set_title(name, fontsize=20)
    ax.axis("off")

plot_cmap("jet")

In [ ]:
plot_cmap("viridis")

Here you can find the full gallery of all the pre-defined colormaps, organized by the types of data they are usually used for: https://matplotlib.org/3.1.0/tutorials/colors/colormaps.html

## Mathtext
Oftentimes, you just simply need that superscript or some other math text in your labels. Matplotlib provides a very easy way to do this for those familiar with LaTeX. Any text that is surrounded by dollar signs will be treated as "[mathtext](http://matplotlib.org/users/mathtext.html#mathtext-tutorial)". Do note that because backslashes are prevelent in LaTeX, it is often a good idea to prepend an `r` to your string literal so that Python will not treat the backslashes as escape characters.

In [ ]:
print(r"a\nb")

In [ ]:
fig, ax = plt.subplots()
ax.scatter([1, 2, 3, 4], [4, 3, 2, 1])
ax.spines['top'].set(visible=False)  # Removing spines so they don't intersect with the title. tight_layout() is not sufficient here.
ax.spines['right'].set(visible=False)
ax.set_title(r'$\sigma_i=\frac{3}{5}$', fontsize=25)
plt.show()

# Limits, Legends and Layouts

In this section, we'll focus on what happens around the edges of the axes:  Ticks, ticklabels, limits, layouts, and legends.

## Legends

As you've seen in some of the examples so far, the X and Y axis can also be labeled, as well as the subplot itself via the title. 

However, another thing you can label is the line/point/bar/etc that you plot.  You can provide a label to your plot, which allows your legend to automatically build itself. 

In [ ]:
fig, ax = plt.subplots()
ax.plot([1, 2, 3, 4], [10, 20, 25, 30])  # Philadelphia
ax.plot([1, 2, 3, 4], [30, 23, 13, 4])  # Boston
ax.set(ylabel='Temperature (deg C)', xlabel='Time', title='A tale of two cities')
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot([1, 2, 3, 4], [10, 20, 25, 30], label='Philadelphia')
ax.plot([1, 2, 3, 4], [30, 23, 13, 4], label='Boston')
ax.set(ylabel='Temperature (deg C)', xlabel='Time', title='A tale of two cities')
ax.legend()
plt.show()

The keyword argument `loc` allows to position the legend at different positions. The `'best'` argument is the default one which automatically chooses the location which overlaps the plot elements as little as possbile.

| Location String | Location Code |
| --- | --- |
| best | 0 |
| upper right | 1 |
| upper left | 2 |
| lower left | 3 |
| lower right | 4 |
| right | 5 |
| center left | 6 |
| center right | 7 |
| lower center | 8 |
| upper center | 9 |
| center | 10 |

In [ ]:
fig, ax = plt.subplots()
ax.plot([1, 2, 3, 4], [10, 20, 25, 30], label='Philadelphia')
ax.plot([1, 2, 3, 4], [30, 23, 13, 4], label='Boston')
ax.set(ylabel='Temperature (deg C)', xlabel='Time', title='A tale of two cities')
ax.legend(loc="center")
plt.show()

## Ticks, Tick Lines, Tick Labels and Tickers
This is a constant source of confusion:

* A Tick is the *location* of a Tick Label.
* A Tick Line is the line that denotes the location of the tick.
* A Tick Label is the text that is displayed at that tick.
* A [`Ticker`](http://matplotlib.org/api/ticker_api.html#module-matplotlib.ticker) automatically determines the ticks for an Axis and formats the tick labels.

[`tick_params()`](http://matplotlib.org/api/axes_api.html#matplotlib.axes.Axes.tick_params) is often used to help configure your tickers.

In [ ]:
fig, ax = plt.subplots()
ax.plot([1, 2, 3, 4], [10, 20, 25, 35])

# Manually set ticks and tick labels *on the x-axis* (note ax.xaxis.set, not ax.set!)
ax.xaxis.set(ticks=[1,2.5,2.7,4], ticklabels=[3, 100, -12, "foo"]) 

# Make the y-ticks a bit longer and go both in and out...
ax.tick_params(axis='y', direction='in', length=10)

plt.show()

In [ ]:
data = [('apples', 2), ('oranges', 3), ('peaches', 1)]
fruit, value = zip(*data)

fig, ax = plt.subplots()
x = np.arange(len(fruit))
ax.bar(x, value, align='center', color='gray')
ax.set(xticks=x, xticklabels=fruit)
plt.show()

# mplot3d
By taking advantage of Matplotlib's z-order layering engine, mplot3d emulates 3D plotting by projecting 3D data into 2D space, layer by layer. While it isn't going to replace any of the true 3D plotting libraries anytime soon, its goal is to allow for Matplotlib users to produce 3D plots with the same amount of simplicity as 2D plots.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D, axes3d

fig, ax = plt.subplots(1, 1, subplot_kw={'projection': '3d'})
X, Y, Z = axes3d.get_test_data(0.05)
ax.plot_wireframe(X, Y, Z, rstride=5, cstride=5)

plt.show()